## 출력 파셔
- llm 출력값을 구조화된 형식으로 변환하고 우리가 원하는 정보만 출력
- 

In [4]:
# !pip install dotenv
from dotenv import load_dotenv

# .env파일에 설정된 보안정보를 읽기.
load_dotenv()

True

In [ ]:
from langchain_openai import ChatOpenAI
#LLM 답변은 보통 문자열인데 원하는 정해진 데이터 형식으로 바꿔주는 역할
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
#itertools를 하나로 이어주는
from itertools import chain
from langchain_core.prompts import PromptTemplate

from langchain_teddynote.messages import stream_response


In [3]:
llm = ChatOpenAI(
    temperature=0,
    model_name= "gpt-4o-mini",
)

In [5]:
# 이메일 예시
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [ ]:
#출력 파셔를 사용하지 않았을 때 답변

prompt = PromptTemplate.from_template(
    "다음 이메일 내용 중 중요한 내용을 추출해 주새요\n\n{email_conversation}"
)

chain = prompt|llm

answer = chain.stream({"email_conversation":email_conversation})

#answer를 화면에 출력 하면서 그 출력 값을 output에 반환
output = stream_response(answer, return_output=True)


중요한 내용 요약:

1. 발신자: 김철수 (바이크코퍼레이션 상무)
2. 수신자: 이은채 (Teddy International)
3. 주제: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안
4. 요청 사항:
   - ZENESIS 모델에 대한 상세 브로슈어 요청 (기술 사양, 배터리 성능, 디자인 정보 포함)
5. 미팅 제안:
   - 날짜: 1월 15일 (다음 주 화요일)
   - 시간: 오전 10시
   - 장소: 귀사 사무실

6. 목적: 협력 가능성 논의 및 유통 전략과 마케팅 계획 구체화.

'중요한 내용 요약:\n\n1. 발신자: 김철수 (바이크코퍼레이션 상무)\n2. 수신자: 이은채 (Teddy International)\n3. 주제: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안\n4. 요청 사항:\n   - ZENESIS 모델에 대한 상세 브로슈어 요청 (기술 사양, 배터리 성능, 디자인 정보 포함)\n5. 미팅 제안:\n   - 날짜: 1월 15일 (다음 주 화요일)\n   - 시간: 오전 10시\n   - 장소: 귀사 사무실\n\n6. 목적: 협력 가능성 논의 및 유통 전략과 마케팅 계획 구체화.'

In [ ]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

#PydanticOutputParser를 가져오는데 pydantic_object에서 앞서 정의한 클래스의 이름을 넣음
parser = PydanticOutputParser(pydantic_object=EmailSummary)

#parser가 만든 출력지시사항 출력
#print(parser.get_format_instructions())

#프롬프트 틀
prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

#프롬프트 format에 들어갈 자리 채워넣기
#get_format_instructions <- json 형식 요구
prompt = prompt.partial(format=parser.get_format_instructions())

chain = prompt | llm

response = chain.stream(
    {
        "email_conversation":email_conversation,
        "question":"이메일 내용 중 주요 내용을 추출해 주세요",
    }
)

#Json 형태로 응답
output = stream_response(response, return_output=True)

```json
{
  "person": "김철수",
  "email": "chulsoo.kim@bikecorporation.me",
  "subject": "\"ZENESIS\" 자전거 유통 협력 및 미팅 일정 제안",
  "summary": "김철수 상무가 이은채 대리님에게 바이크코퍼레이션의 자전거 'ZENESIS'에 대한 브로슈어 요청과 협력 논의를 위한 미팅 제안을 보냈습니다.",
  "date": "1월 15일 오전 10시"
}
```

In [11]:
#pydanticOutputParser 결과 파싱
structed_output = parser.parse(output)
print(structed_output)

person='김철수' email='chulsoo.kim@bikecorporation.me' subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안' summary="김철수 상무가 이은채 대리님에게 바이크코퍼레이션의 자전거 'ZENESIS'에 대한 브로슈어 요청과 협력 논의를 위한 미팅 제안을 보냈습니다." date='1월 15일 오전 10시'


In [ ]:
# : 프롬프트주입 방식을 사용하는, chain 재구성
#원래는 llm 답변까지여는데 지금은parser 추가해서 결과 변환?
chain = prompt | llm | parser

response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

response

EmailSummary(person='김철수', email='chulsoo.kim@bikecorporation.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary="김철수 상무가 이은채 대리님에게 'ZENESIS' 자전거에 대한 브로슈어 요청과 협력 논의를 위한 미팅 제안을 보냈습니다.", date='1월 15일 오전 10시')

## 02. with_structured_output() 바인딩 page 172
- 모델에 스키마를 강제하므로, 파싱오류가 거의 없음.
- 프롬프트 토근 절약.
- 속성이 여러개인 복잡한 구조체에 적합.
- 엔티티 추출, API 파라미터 매핑, RAG 구조화 데이터.

In [14]:
llm.invoke("이탈리아 수도 뭐야")

AIMessage(content='이탈리아의 수도는 로마(Roma)입니다. 로마는 이탈리아의 역사적, 문화적 중심지로 유명하며, 많은 유적지와 관광 명소가 있습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 42, 'prompt_tokens': 13, 'total_tokens': 55, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_ba6e3550a6', 'id': 'chatcmpl-EP0S86bYo7y0wvxILSw8F5BTKBUWU', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0ae2e-0af4-7f31-9ab7-f8a7001af242-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 42, 'total_tokens': 55, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details

In [21]:
#네이티브 API 방식: 모델에 스키마 강제

llm_with_structured = ChatOpenAI(
    temperature=0, model="gpt-4o-mini"
).with_structured_output(EmailSummary) #답변할때 정의한 구조에 맞춰서 결과를 반환

answer=llm_with_structured.invoke(email_conversation)
answer


EmailSummary(person='이은채', email='eunchae@teddyinternational.me', subject='"ZENESIS" 자전거 유통 협력 및 미팅 일정 제안', summary="김철수 상무가 이은채 대리에게 바이크코퍼레이션과 'ZENESIS' 자전거의 유통 협력에 대해 논의하고자 미팅을 제안하며, 제품에 대한 상세한 브로슈어 요청.", date='2024-01-08')

## 쉼표로 구분된 리스트 출력파서 commaSeparatedListOutputParser

- list 객체로 파싱
- 쉼표로 구분 출력
- 낮은 토큰 소비 & 빠른 속도
- 약간의 파싱 실패율 있음
- 연관키워드추출, 태그생성, 카테고리 목록 추출 등에 사용.

In [18]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate

#콤마로 구분된 리스트 출력 파서 초기화
output_parser = CommaSeparatedListOutputParser()

#출력 형식 지침 가져오기
format_instructions = output_parser.get_format_instructions()

print(format_instructions)

#프롬프트 템플릿 설정
prompt = PromptTemplate(
    #주제에 대한 다섯가지(영어)를 나열하라는 템플릿
    template="List five {subject}.\n{format_instructions}",
    input_variables=["subject"], #입력 변수로 'subject'사용
    # 부분 변수로 형식 지침 사용
    partial_variables={"format_instructions":format_instructions},
)

# 프롬프트2 템플릿 설정
prompt2 = PromptTemplate(
    # 주제에 대한 다섯가지(한글)를 나열하라는 템플릿
    template="다섯가지 {subject}.\n{format_instructions}",
    input_variables=["subject"],  # 입력 변수로 'subject' 사용
    # 부분 변수로 형식 지침 사용
    partial_variables={"format_instructions": format_instructions},
)

# ChatOpenAI 모델 초기화
model = ChatOpenAI(temperature=0)

# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain = prompt | model | output_parser

chain.invoke({"subject":"호주 관광명소"})



Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


['시드니 오페라하우스', '그레이트 오션 로드', '울룰루', '그레이트 베리어 리프', '블루 마운틴즈']

In [19]:
# ChatOpenAI 모델 초기화
model = ChatOpenAI(temperature=0)

# 프롬프트, 모델, 출력 파서를 연결하여 체인 생성
chain2 = prompt2 | model | output_parser

# "인도 관광명소"에 대한 체인 호출 실행
chain2.invoke({"subject": "인도 관광명소"})

['타지마할', '자이푸르', '고아', '코찌코데', '바라나시']

In [20]:
# 스트림을 순회.
for s in chain.stream({"subject": "대한민국 관광명소"}):
    print(s)  

['경복궁']
['인사동']
['부산 해운대해수욕장']
['제주도']
['남산타워']
